# Phase 1 — 실시간 토론 채점 데모
PCT(Perceptual Control Theory) 기반 턴제 독립 채점 엔진

**실행 순서:** 셀을 위에서 아래로 순서대로 실행하세요.

## 1. 환경 설치

In [ ]:
!pip install transformers accelerate torch -q

## 2. 레포 클론 및 경로 설정

In [ ]:
import os, sys

REPO_URL = "https://github.com/shinjipark22/SJU-Capstone-Multi-Agent-Debater-AI.git"
BRANCH   = "feature/ymj-scoring"
REPO_DIR = "/content/debate-ai"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull origin {BRANCH}

sys.path.insert(0, REPO_DIR)
print("경로 설정 완료:", REPO_DIR)

## 3. Qwen 7B 모델 로드 확인
> GPU 런타임(T4 이상)으로 전환 후 실행하세요.  
> 런타임 → 런타임 유형 변경 → T4 GPU

In [ ]:
import torch
print("GPU 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

# 모델 로드 (최초 1회, 약 3~5분 소요)
from src.phase1.extractor import extract_rg, judge_turn
print("\n모델 로드 시작...")
r, g = extract_rg("테스트 발언입니다.")
print(f"테스트 추출 완료 — r={r}, g={g}")

## 4. 예시 데이터 — 1:1 토론 (3턴)

In [ ]:
from src.phase1.scoring import DebateScorer, print_turn_result

agents = [
    {"agent_id": "user",    "stance": "PRO"},
    {"agent_id": "agent_1", "stance": "CON"},
]
scorer = DebateScorer(agents)

debates = [
    (
        "opening",
        ("user",
         "AI 규제는 반드시 필요합니다. 딥페이크 범죄가 2023년 전년 대비 300% 급증했고, "
         "생성형 AI로 인한 허위정보 유포 피해가 심각합니다. "
         "EU AI Act는 시행 6개월 만에 딥페이크 관련 피해 신고를 42% 감소시켰습니다. "
         "규제 없이는 피해자 보호가 불가능합니다."),
        ("agent_1",
         "규제는 기술 혁신을 심각하게 저해합니다. "
         "규제가 없는 미국은 AI 분야 세계 특허 1위, 투자 유치 1위를 유지하고 있습니다. "
         "반면 강한 규제를 택한 국가들은 AI 스타트업 이탈 현상이 뚜렷합니다. "
         "혁신을 막는 규제는 결국 국가 경쟁력 약화로 이어집니다."),
    ),
    (
        "chained_rebuttal",
        ("user",
         "상대측은 혁신을 말하지만, 규제 없는 혁신은 사회적 재앙입니다. "
         "ChatGPT 출시 이후 피싱 이메일이 1,265% 증가했다는 SlashNext 보고서가 있습니다. "
         "혁신과 안전은 양립 가능하며, EU는 규제와 동시에 AI 투자도 확대하고 있습니다."),
        ("agent_1",
         "찬성측이 인용한 EU 사례는 왜곡입니다. "
         "EU AI Act 시행 이후 오히려 유럽 AI 기업의 미국·아시아 이전이 가속화되었습니다. "
         "규제가 범죄를 막는다는 주장도 근거가 없습니다. "
         "규제는 합법적 기업만 옭아매고, 실제 범죄자는 이를 무시합니다."),
    ),
    (
        "free_rebuttal",
        ("user",
         "상대측 주장에는 심각한 논리 오류가 있습니다. "
         "기업 이전 데이터는 AI Act 시행 전부터 진행된 글로벌 트렌드이며, "
         "AI Act와의 인과관계를 증명하지 못합니다. "
         "반면 규제의 범죄 억지 효과는 OECD 15개국 비교 연구에서 통계적으로 유의미하게 확인되었습니다."),
        ("agent_1",
         "찬성측이 언급한 OECD 연구는 존재하지 않는 자료입니다. 출처를 밝히시기 바랍니다. "
         "AI 규제의 실효성에 대한 학계 합의는 아직 없으며, "
         "오히려 자율규제와 기술표준화가 더 효과적이라는 연구가 다수입니다. "
         "규제 만능주의는 위험한 발상입니다."),
    ),
]

print("=" * 55)
print("        논제: AI 규제는 필요한가")
print("=" * 55)

for phase, first, second in debates:
    print()
    result = scorer.process_pair(phase=phase, first_speech=first, second_speech=second)
    print_turn_result(result)

## 5. 예시 데이터 — 2:2 토론 (입론 2턴)

In [ ]:
agents_2v2 = [
    {"agent_id": "pro_1", "stance": "PRO"},
    {"agent_id": "pro_2", "stance": "PRO"},
    {"agent_id": "con_1", "stance": "CON"},
    {"agent_id": "user",  "stance": "CON"},
]
scorer_2v2 = DebateScorer(agents_2v2)

speeches_2v2 = [
    ("pro_1",
     "사형제 폐지는 국가가 인간의 생명을 박탈할 권리가 없다는 인권적 관점에서 필요합니다. "
     "국제앰네스티에 따르면 2023년 사형제를 폐지한 국가가 106개국으로 역대 최다입니다."),
    ("con_1",
     "사형제는 극악한 범죄에 대한 사회 방위 수단입니다. "
     "종신형은 국민 세금으로 흉악범을 부양하는 구조이며, 피해자 가족의 응보 감정도 외면할 수 없습니다."),
    ("pro_2",
     "사형제의 오판 위험성은 치명적입니다. "
     "미국에서만 1973년 이후 190명 이상이 사형 집행 전 무죄로 석방되었습니다. "
     "돌이킬 수 없는 형벌은 사법 오류를 영원히 바로잡을 수 없게 만듭니다."),
    ("user",
     "폐지론자들은 억지 효과가 없다고 주장하지만, 이는 편향된 연구 선택입니다. "
     "Hashem Dezhbakhsh 등의 연구에 따르면 사형 집행 1건당 살인 억지 효과가 평균 18건에 달합니다. "
     "오판 문제는 사형제 폐지가 아닌 사법 절차 강화로 해결해야 합니다."),
]

print("=" * 55)
print("        논제: 사형제 폐지는 필요한가 (2:2)")
print("=" * 55)

results = scorer_2v2.process_phase_speeches(
    phase="opening",
    speeches=speeches_2v2,
    on_pair_complete=lambda r: (print(), print_turn_result(r)),
)

## 6. 결과 DataFrame 확인

In [ ]:
import pandas as pd

rows = []
for result in results:
    for ss in result.speeches:
        rows.append({
            "turn":      result.turn_index + 1,
            "phase":     result.phase,
            "agent_id":  ss.agent_id,
            "stance":    ss.stance,
            "mag (r)":   ss.magnitude,
            "ref":       ss.reference,
            "gain (g)":  ss.gain,
            "o":         round(ss.o, 4),
            "v":         round(result.v, 4),
            "dominance": result.dominance,
            "gap":       round(result.dominance_gap, 4),
            "Qwen 판정": result.dominance_judgment,
        })

df = pd.DataFrame(rows)
df